# 01 - Setup and Data Preparation

This notebook initializes the Colab environment for the Deep Learning project and runs the data preprocessing scripts. It sets up the repository, installs dependencies, maps Google Drive folders, and processes the raw S3DIS data into `.npy` format.

### 1. Setup & Mount Drive

In [2]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

import os
REPO_DIR = '/content/Deep_learning_project'
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/Gandata/Deep_learning_project.git {REPO_DIR}
else:
    %cd {REPO_DIR}
    !git pull origin dev

%cd {REPO_DIR}

# Install uv and dependencies
!pip install uv
!uv pip install --system -e .

Mounted at /content/drive
Cloning into '/content/Deep_learning_project'...
remote: Enumerating objects: 98, done.
remote: Counting objects: 100% (98/98), done.
remote: Compressing objects: 100% (74/74), done.
remote: Total 98 (delta 43), reused 65 (delta 22), pack-reused 0 (from 0)
Receiving objects: 100% (98/98), 30.70 MiB | 38.76 MiB/s, done.
Resolving deltas: 100% (43/43), done.
/content/Deep_learning_project
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.8/24.8 MB 62.2 MB/s eta 0:00:00
Using Python 3.12.13 environment at: /usr
Resolved 97 packages in 960ms
Prepared 13 packages in 12.30s
Uninstalled 2 packages in 11ms
Installed 13 packages in 260ms
 + addict==2.4.0
 + comm==0.2.3
 + configargparse==1.7.5
 + dash==4.1.0
 + deep-learning-project==0.1.0 (from file:///content/Deep_learning_project)
 + ftfy==6.3.1
 - ipywidgets==7.7.1
 + ipywidgets==8.1.8
 + jedi==0.19.2
 + open-clip-torch==3.3.0
 + open3d==0.19.0
 + pyquaternion==0.9.9
 + retrying==1.4.2
 - widgetsnbextension==3.6.10
 +

### 2. Symlink Data & Checkpoints
We map the Drive folders directly into the repository so our scripts can find them easily.

In [3]:
# Symlink Drive data
!rm -f data checkpoints features pretrained
!ln -sf /content/drive/MyDrive/DL_Project/data ./data
!ln -sf /content/drive/MyDrive/DL_Project/checkpoints ./checkpoints
!ln -sf /content/drive/MyDrive/DL_Project/features ./features

In [1]:
!find "/content/drive/MyDrive/DL_Project" -maxdepth 10 -path "/content/drive/MyDrive/DL_Project/data/s3dis_raw/Stanford3dDataset_v1.2_Aligned_Version" -prune -o  -print

find: ‘/content/drive/MyDrive/DL_Project’: No such file or directory


### 3. Preprocess S3DIS Dataset
Make sure you have downloaded the raw S3DIS dataset into `/content/drive/MyDrive/DL_Project/data/s3dis_raw/`.
This script will process it into `.npy` files required by our `S3DISDataset` class.
First, we unzip it (if it hasn't been unzipped already), and then process it into `.npy` files required by our `S3DISDataset` class.

In [4]:
import os

zip_path = "data/s3dis_raw/Stanford3dDataset_v1.2_Aligned_Version.zip"
extract_dir = "data/s3dis_raw/Stanford3dDataset_v1.2_Aligned_Version"

# Unzip if the folder doesn't exist yet
if not os.path.exists(extract_dir):
    print("Unzipping dataset... this may take a few minutes.")
    !unzip -q {zip_path} -d data/s3dis_raw/
    print("Unzip complete.")
else:
    print("Dataset already unzipped.")

Dataset already unzipped.


In [5]:
# Run S3DIS preprocessing
!uv run scripts/prepare_s3dis.py --input {extract_dir} --output data/s3dis_processed

Using CPython 3.11.15
Creating virtual environment at: .venv
Installed 99 packages in 1.00s
18:07:43  INFO      Label map saved → data/s3dis_processed/label_map.json
18:07:43  WARNING   Area not found, skipping: data/s3dis_raw/Stanford3dDataset_v1.2_Aligned_Version/Area_1
18:07:44  INFO      
──────────────────────────────────────────────────
18:07:44  INFO      Processing Area_2 ...
18:07:44  INFO        Processing room: WC_1
18:08:00  INFO          → saved 610,183 points  (WC_1.npy)
18:08:01  INFO        Processing room: WC_2
18:08:28  INFO          → saved 859,076 points  (WC_2.npy)
18:08:28  INFO        Processing room: auditorium_1
18:08:39  INFO          → saved 2,451,238 points  (auditorium_1.npy)
18:08:39  INFO        Processing room: auditorium_2
18:09:00  INFO          → saved 3,986,575 points  (auditorium_2.npy)
18:09:00  INFO        Processing room: conferenceRoom_1
18:09:54  INFO          → saved 1,203,377 points  (conferenceRoom_1.npy)
18:09:54  INFO        Processing roo

### 4. Process Polycam Scans
If you have uploaded any Polycam `.ply` scans to `/content/drive/MyDrive/DL_Project/data/polycam/`, run this script to convert them into our standard `.npy` format.

In [ ]:
# Run Polycam export
!uv run scripts/export_polycam.py --input_dir data/polycam --out_dir data/polycam_processed